# Task 8B — Native physical-signal readiness

This notebook uses licensed data extracted into local runtime storage, creates a reviewable inventory, builds the Task 8B manifest and matched TIFF view, runs source/nuisance gates, validates PRNU without binary labels, and records a fail-closed retention decision. In Colab, the preparation cell downloads the verified `task8b_manifest.tar.gz.upload-*` chunks from the configured shared Drive folder, reconstructs and safely extracts them into `/content/hackathon_data/raw/task8b`, and verifies every `sources.csv` path plus populated PREMIER `N1` and `N2` trees. For another environment, stage `premier/` and `genimage_ai/` under `hackathon_data/raw/task8b/`, or set `CYA_TASK8B_DATA_ROOT` to a directory containing them. The source images are intentionally not stored in Git. The notebook does not modify RINE, read the competition final test, or enable chromatic aberration automatically.

In [1]:
!git clone https://github.com/maxi-cmyk/cya-techjam26.git

fatal: destination path 'cya-techjam26' already exists and is not an empty directory.


In [ ]:
from pathlib import Path
import csv
import hashlib
import os
import shutil
import tarfile
import unicodedata

import google.auth
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Authenticate with the Google account that can access the shared folder.
auth.authenticate_user()

DRIVE_FOLDER_ID = "1ANvP41AjiTztc0Hhv3rT-XjKguG0YNUp"
TARGET = Path("/content/hackathon_data/raw/task8b")

# Safety limit: expected extracted data is roughly 2 GB.
MAX_BYTES = 3 * 1024**3

FOLDER_MIME = "application/vnd.google-apps.folder"
SHORTCUT_MIME = "application/vnd.google-apps.shortcut"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
REQUIRED_FOLDERS = {"premier", "genimage_ai"}
ARCHIVE_PREFIX = "task8b_manifest.tar.gz.upload-"
EXPECTED_CHUNK_NAMES = [
    f"{ARCHIVE_PREFIX}a{chr(ord('a') + index)}"
    for index in range(23)
]
EXPECTED_ARCHIVE_BYTES = 2_135_569_689
EXPECTED_ARCHIVE_SHA256 = (
    "e4007820df21ca640a2a3fa5b25a315b0b9476834beab0995fcb36bb99b56b25"
)

credentials, _ = google.auth.default()
drive = build(
    "drive",
    "v3",
    credentials=credentials,
    cache_discovery=False,
)


def normalize_drive_name(name):
    """Normalize Drive folder names for reliable matching."""

    return (
        unicodedata.normalize("NFKC", name)
        .strip()
        .casefold()
        .replace(" ", "_")
    )


def safe_name(name):
    """Reject unsafe filenames before writing locally."""

    normalized = unicodedata.normalize("NFKC", name).strip()

    if (
        not normalized
        or normalized in {".", ".."}
        or Path(normalized).name != normalized
    ):
        raise RuntimeError(f"Unsafe Drive filename: {name!r}")

    return normalized


def list_children(folder_id):
    """List all children of one authenticated Drive folder."""

    results = []
    page_token = None

    while True:
        response = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields=(
                "nextPageToken,"
                "files("
                "id,"
                "name,"
                "mimeType,"
                "size,"
                "shortcutDetails(targetId,targetMimeType)"
                ")"
            ),
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()

        for original_item in response.get("files", []):
            item = dict(original_item)

            # Resolve Drive shortcuts while preserving the shortcut's name.
            if item["mimeType"] == SHORTCUT_MIME:
                details = item.get("shortcutDetails", {})
                target_id = details.get("targetId")
                target_mime = details.get("targetMimeType")

                if not target_id or not target_mime:
                    raise RuntimeError(
                        f"Could not resolve Drive shortcut: {item['name']}"
                    )

                item["id"] = target_id
                item["mimeType"] = target_mime

                # Fetch the target size when the shortcut points to a file.
                if target_mime != FOLDER_MIME:
                    metadata = drive.files().get(
                        fileId=target_id,
                        fields="id,size",
                        supportsAllDrives=True,
                    ).execute()
                    item["size"] = metadata.get("size", "0")

            results.append(item)

        page_token = response.get("nextPageToken")
        if not page_token:
            break

    return results


def download_drive_item(item, destination):
    """Download one Drive file with size checks and rerun safety."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(item.get("size", -1) or -1)

    if (
        destination.is_file()
        and expected_size >= 0
        and destination.stat().st_size == expected_size
    ):
        return False

    temporary = destination.with_suffix(destination.suffix + ".part")
    request = drive.files().get_media(
        fileId=item["id"],
        supportsAllDrives=True,
    )

    with temporary.open("wb") as output:
        downloader = MediaIoBaseDownload(
            output,
            request,
            chunksize=16 * 1024 * 1024,
        )
        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=3)

    if expected_size >= 0 and temporary.stat().st_size != expected_size:
        actual_size = temporary.stat().st_size
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Size verification failed for {destination.name}: "
            f"expected {expected_size}, downloaded {actual_size}"
        )

    temporary.replace(destination)
    return True


# The supplied Drive folder directly contains premier/ and genimage_ai/.
payload_children = list_children(DRIVE_FOLDER_ID)

archive_chunks = sorted(
    (item for item in payload_children if item["name"].startswith(ARCHIVE_PREFIX)),
    key=lambda item: item["name"],
)
archive_mode = bool(archive_chunks)

if archive_mode:
    chunk_names = [item["name"] for item in archive_chunks]
    if chunk_names != EXPECTED_CHUNK_NAMES:
        missing = sorted(set(EXPECTED_CHUNK_NAMES) - set(chunk_names))
        unexpected = sorted(set(chunk_names) - set(EXPECTED_CHUNK_NAMES))
        raise RuntimeError(
            f"Task 8B archive chunks are incomplete. Missing: {missing}; "
            f"unexpected: {unexpected}"
        )

    remote_archive_bytes = sum(
        int(item.get("size", 0) or 0) for item in archive_chunks
    )
    if remote_archive_bytes != EXPECTED_ARCHIVE_BYTES:
        raise RuntimeError(
            f"Archive size mismatch: expected {EXPECTED_ARCHIVE_BYTES}, "
            f"found {remote_archive_bytes}"
        )

    TARGET.mkdir(parents=True, exist_ok=True)
    completion_marker = TARGET / ".task8b_archive_sha256"

    def staged_inventory_complete():
        inventory = TARGET / "sources.csv"
        if not inventory.is_file():
            return False
        with inventory.open(newline="", encoding="utf-8") as source_file:
            rows = list(csv.DictReader(source_file))
        return len(rows) == 1_280 and all(
            (TARGET / row["relative_path"]).is_file() for row in rows
        )

    already_staged = (
        completion_marker.is_file()
        and completion_marker.read_text(encoding="utf-8").strip()
        == EXPECTED_ARCHIVE_SHA256
        and all((TARGET / name).is_dir() for name in REQUIRED_FOLDERS)
        and staged_inventory_complete()
    )

    if already_staged:
        print("Verified Task 8B archive already extracted; skipping download.")
    else:
        chunk_dir = Path("/content/task8b_archive_chunks")
        chunk_dir.mkdir(parents=True, exist_ok=True)

        for index, item in enumerate(archive_chunks, start=1):
            changed = download_drive_item(item, chunk_dir / item["name"])
            status = "downloaded" if changed else "already present"
            print(f"Chunk {index:02d}/{len(archive_chunks)}: {status}")

        archive_path = Path("/content/task8b_manifest.tar.gz")
        digest = hashlib.sha256()
        written = 0

        with archive_path.open("wb") as combined:
            for item in archive_chunks:
                part_path = chunk_dir / item["name"]
                with part_path.open("rb") as part:
                    while block := part.read(16 * 1024 * 1024):
                        combined.write(block)
                        digest.update(block)
                        written += len(block)

        if written != EXPECTED_ARCHIVE_BYTES:
            raise RuntimeError(
                f"Combined archive size mismatch: expected "
                f"{EXPECTED_ARCHIVE_BYTES}, wrote {written}"
            )
        if digest.hexdigest() != EXPECTED_ARCHIVE_SHA256:
            raise RuntimeError("Combined archive SHA-256 verification failed.")

        target_root = TARGET.resolve()
        with tarfile.open(archive_path, mode="r:gz") as archive:
            members = archive.getmembers()
            for member in members:
                destination = (TARGET / member.name).resolve()
                if os.path.commonpath([target_root, destination]) != str(target_root):
                    raise RuntimeError(f"Unsafe archive member: {member.name}")
                if not (member.isfile() or member.isdir()):
                    raise RuntimeError(f"Unsupported archive member: {member.name}")
            try:
                archive.extractall(TARGET, members=members, filter="data")
            except TypeError:
                archive.extractall(TARGET, members=members)

        completion_marker.write_text(EXPECTED_ARCHIVE_SHA256 + "\n", encoding="utf-8")
        archive_path.unlink(missing_ok=True)
        shutil.rmtree(chunk_dir)
        print("Verified and extracted the complete Task 8B archive.")

payload_folders = {
    normalize_drive_name(item["name"]): item
    for item in payload_children
    if item["mimeType"] == FOLDER_MIME
}

missing_folders = REQUIRED_FOLDERS - set(payload_folders)

if missing_folders:
    raise RuntimeError(
        f"Missing required Drive folders: {sorted(missing_folders)}. "
        f"Found folders: {sorted(payload_folders)}"
    )

print("Drive folders found:", sorted(payload_folders))


# Collect only supported images. ZIP and TAR archives are deliberately skipped.
remote_files = []
visited_folders = set()


def collect_images(folder_id, relative_root):
    if folder_id in visited_folders:
        return

    visited_folders.add(folder_id)

    for item in list_children(folder_id):
        name = safe_name(item["name"])
        relative_path = relative_root / name

        if item["mimeType"] == FOLDER_MIME:
            collect_images(item["id"], relative_path)
            continue

        if Path(name).suffix.lower() in IMAGE_EXTENSIONS:
            remote_files.append((item, relative_path))


if not archive_mode:
    for folder_name in sorted(REQUIRED_FOLDERS):
        folder = payload_folders[folder_name]
        collect_images(folder["id"], Path(folder_name))


# Include the already-reviewed inventory when present.
if not archive_mode:
    for item in payload_children:
        if normalize_drive_name(item["name"]) == "sources.csv":
            remote_files.append((item, Path("sources.csv")))


if not remote_files and not archive_mode:
    raise RuntimeError(
        "No supported images were found in the Drive folders."
    )

total_bytes = sum(
    int(item.get("size", 0) or 0)
    for item, _ in remote_files
)

if archive_mode:
    print(f"Archive chunks selected: {len(archive_chunks):,}")
    print(f"Archive size: {EXPECTED_ARCHIVE_BYTES / 1024**3:.2f} GiB")
else:
    print(f"Files selected: {len(remote_files):,}")
    print(f"Download size: {total_bytes / 1024**3:.2f} GiB")

if total_bytes > MAX_BYTES:
    raise RuntimeError(
        f"Refusing a {total_bytes / 1024**3:.2f} GiB download. "
        "Inspect the selected Drive content before increasing MAX_BYTES."
    )


TARGET.mkdir(parents=True, exist_ok=True)

downloaded = 0
skipped = 0

for index, (item, relative_path) in enumerate(remote_files, start=1):
    destination = TARGET / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)

    expected_size = int(item.get("size", -1) or -1)

    # Make the cell safe to rerun.
    if (
        destination.is_file()
        and expected_size >= 0
        and destination.stat().st_size == expected_size
    ):
        skipped += 1
        continue

    temporary = destination.with_suffix(destination.suffix + ".part")

    request = drive.files().get_media(
        fileId=item["id"],
        supportsAllDrives=True,
    )

    with temporary.open("wb") as output:
        downloader = MediaIoBaseDownload(
            output,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=3)

    if expected_size >= 0 and temporary.stat().st_size != expected_size:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Size verification failed for {relative_path}: "
            f"expected {expected_size}, downloaded "
            f"{temporary.stat().st_size if temporary.exists() else 'invalid'}"
        )

    temporary.replace(destination)
    downloaded += 1

    if index % 50 == 0 or index == len(remote_files):
        print(f"Processed {index:,}/{len(remote_files):,} files")


# Final validation.
counts = {}

for folder_name in sorted(REQUIRED_FOLDERS):
    folder = TARGET / folder_name

    if not folder.is_dir():
        raise RuntimeError(f"Staging failed; missing directory: {folder}")

    counts[folder_name] = sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in folder.rglob("*")
    )

    if counts[folder_name] == 0:
        raise RuntimeError(
            f"Staging failed; no supported images found under {folder}"
        )


inventory_path = TARGET / "sources.csv"
if not inventory_path.is_file():
    raise RuntimeError(f"Staging failed; missing inventory: {inventory_path}")

with inventory_path.open(newline="", encoding="utf-8") as source_file:
    inventory_rows = list(csv.DictReader(source_file))

if len(inventory_rows) != 1_280:
    raise RuntimeError(
        f"Expected 1,280 inventory rows, found {len(inventory_rows):,}"
    )

missing_inventory_files = [
    row["relative_path"]
    for row in inventory_rows
    if not (TARGET / row["relative_path"]).is_file()
]
if missing_inventory_files:
    raise RuntimeError(
        f"Staging failed; {len(missing_inventory_files):,} inventory files are "
        f"missing. First missing path: {missing_inventory_files[0]}"
    )

premier_split_counts = {
    split: sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in (TARGET / "premier" / split).rglob("*")
    )
    for split in ("N1", "N2")
}
if any(count == 0 for count in premier_split_counts.values()):
    raise RuntimeError(
        f"PREMIER extraction incomplete; expected populated N1 and N2: "
        f"{premier_split_counts}"
    )


# The following notebook cells will now use this exact directory.
os.environ["CYA_TASK8B_DATA_ROOT"] = str(TARGET)

print()
print("Downloaded:", downloaded)
print("Already present:", skipped)
print("Image counts:", counts)
print("PREMIER split counts:", premier_split_counts)
print("Task 8B data staged at:", TARGET)

In [ ]:
from pathlib import Path
import subprocess

def checkout_present(path):
    return (path / 'configs/colab.json').is_file() and (path / 'Makefile').is_file()

cwd = Path.cwd()
running_from_checkout = checkout_present(cwd) or checkout_present(cwd.parent)
cloud_checkout = Path('/content/cya-techjam26')
if not running_from_checkout and Path('/content').is_dir():
    if (cloud_checkout / '.git').is_dir():
        subprocess.run(['git', '-C', str(cloud_checkout), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(
            ['git', 'clone', 'https://github.com/maxi-cmyk/cya-techjam26.git', str(cloud_checkout)],
            check=True,
        )

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

def find_project_root():
    configured = os.environ.get('CYA_PROJECT_ROOT')
    starts = [Path(configured)] if configured else []
    starts.extend([Path.cwd(), Path.cwd().parent, Path('/content/cya-techjam26')])
    checked = set()
    for start in starts:
        for candidate in (start, *start.parents):
            resolved = candidate.resolve()
            if resolved in checked:
                continue
            checked.add(resolved)
            if (resolved / 'configs/colab.json').is_file() and (resolved / 'Makefile').is_file():
                return resolved
    raise RuntimeError(
        'Could not locate the cya-techjam26 checkout. Open this notebook from the repository, '
        'or set CYA_PROJECT_ROOT to its absolute path.'
    )

PROJECT_ROOT = find_project_root()
SOURCE_MODE = 'local'  # 'local' or 'private_drive'
PRIVATE_DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/cya-techjam26-data')
TASK8B_DATA_ROOT_OVERRIDE = os.environ.get('CYA_TASK8B_DATA_ROOT', '').strip()
if TASK8B_DATA_ROOT_OVERRIDE:
    LOCAL_TASK8B = Path(TASK8B_DATA_ROOT_OVERRIDE).expanduser().resolve()
    LOCAL_DATA_ROOT = LOCAL_TASK8B.parents[1]
else:
    LOCAL_DATA_ROOT = (
        Path('/content/hackathon_data')
        if PROJECT_ROOT == Path('/content/cya-techjam26')
        else PROJECT_ROOT / 'hackathon_data'
    )
    LOCAL_TASK8B = LOCAL_DATA_ROOT / 'raw/task8b'
LOCAL_ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
PRIVATE_DRIVE_TASK8B = PRIVATE_DRIVE_DATA_ROOT / 'raw/task8b'

assert SOURCE_MODE in {'local', 'private_drive'}, 'Invalid SOURCE_MODE'
print('Project root:', PROJECT_ROOT)
print('Task 8B data root:', LOCAL_TASK8B)

In [ ]:
if SOURCE_MODE == 'private_drive':
    if not PRIVATE_DRIVE_TASK8B.is_dir():
        raise FileNotFoundError(
            f'Missing personal Drive data: {PRIVATE_DRIVE_TASK8B}. Mount Drive or update '
            'PRIVATE_DRIVE_DATA_ROOT before continuing.'
        )
    LOCAL_TASK8B.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(PRIVATE_DRIVE_TASK8B, LOCAL_TASK8B, dirs_exist_ok=True)

required_directories = [LOCAL_TASK8B / 'premier', LOCAL_TASK8B / 'genimage_ai']
missing_directories = [path for path in required_directories if not path.is_dir()]
if missing_directories:
    missing = '\n'.join(f'  - {path}' for path in missing_directories)
    raise FileNotFoundError(
        'The code checkout is ready, but Task 8B image data is not included in Git.\n'
        f'Missing directories:\n{missing}\n\n'
        'Stage the extracted data at those paths, or set CYA_TASK8B_DATA_ROOT to a '
        'directory that directly contains premier/ and genimage_ai/. Only the extracted '
        'PREMIER sample (about 1.9 GB) and Tiny-GenImage sample (about 44 MB) are needed; '
        'the original PREMIER tar.gz files are not required.'
    )
print('Using Task 8B source data at:', LOCAL_TASK8B)

In [ ]:
environment = os.environ.copy()
environment['DATA_ROOT'] = str(LOCAL_DATA_ROOT)
environment['ARTIFACT_ROOT'] = str(LOCAL_ARTIFACT_ROOT)
inventory = LOCAL_TASK8B / 'sources.csv'
if not inventory.is_file():
    subprocess.run(['make', 'task8b-inventory'], cwd=PROJECT_ROOT, env=environment, check=True)
    raise RuntimeError(
        'A draft sources.csv was created. Review it and inventory_preparation.json, '
        'correct any metadata, then rerun this cell.'
    )
subprocess.run(['make', 'task8b-prepare'], cwd=PROJECT_ROOT, env=environment, check=True)

In [ ]:
readiness_path = LOCAL_ARTIFACT_ROOT / 'task8b/audits/readiness_report.json'
readiness = json.loads(readiness_path.read_text(encoding='utf-8'))
print(json.dumps({key: readiness[key] for key in ('source_ready', 'training_ready', 'prnu_reference', 'chromatic_aberration')}, indent=2))
assert readiness['source_ready'], 'Source readiness failed; inspect readiness_report.json'
if readiness['prnu_reference']['ready']:
    subprocess.run(['make', 'task8b-prnu-references'], cwd=PROJECT_ROOT, env=environment, check=True)
else:
    print('PRNU references remain blocked by device/image coverage.')
subprocess.run(['make', 'task8b-matched'], cwd=PROJECT_ROOT, env=environment, check=True)
subprocess.run(['make', 'task8b-prnu-validate'], cwd=PROJECT_ROOT, env=environment, check=True)
subprocess.run(['make', 'task8b-decision'], cwd=PROJECT_ROOT, env=environment, check=True)
decision_path = LOCAL_ARTIFACT_ROOT / 'task8b/reports/retention_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print(json.dumps(decision, indent=2))
if not decision['fusion_training_eligible']:
    print('Task 8B is complete with no fusion run; no physical estimator passed its independent gate.')

In [ ]:
local_results = LOCAL_ARTIFACT_ROOT / 'task8b'
drive_results = DRIVE_ARTIFACT_ROOT / 'task8b'
if Path('/content/drive/MyDrive').is_dir():
    drive_results.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    print('Synced Task 8B artifacts to:', drive_results)
else:
    print('Drive is not mounted; artifacts remain at:', local_results)